# 🇰🇭 Khmer Whisper LoRA Fine-Tuning Notebook
### បង្រៀន AI Whisper ភាសាខ្មែរ (Khmer & Khmer-English Tech Speech) ដោយឥតគិតថ្លៃលើ Google Colab T4 GPU

Notebook នេះត្រូវបានរៀបចំឡើងយ៉ាងពិសេសសម្រាប់ **KhmerSub AI** ដើម្បី៖
1. បង្រៀន (Fine-tune) Whisper Small / Medium ជាមួយ LoRA (PEFT) លើទិន្នន័យខ្មែរ
2. កាត់បន្ថយ CER (Character Error Rate) លើពាក្យបច្ចេកវិទ្យា (Code-Switching Tech terms)
3. Export ទៅជា **CTranslate2 (int8)** សម្រាប់ Run លើ Server CPU ដោយមិនបាច់ចំណាយថ្លៃ GPU!

## ជំហានទី ១: Install Dependencies & Check GPU

In [ ]:
!nvidia-smi
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers datasets peft evaluate jiwer soundfile librosa bitsandbytes ctranslate2 accelerate

## ជំហានទី ២: Clone Repository ឬរៀបចំ Dataset

In [ ]:
import os
import unicodedata
import json
from pathlib import Path

# បង្កើត Folder សម្រាប់ផ្ទុក Dataset
DATASET_DIR = Path("./khmer_dataset")
DATASET_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR = DATASET_DIR / "audio"
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

print("Ready for dataset ingestion!")

## ជំហានទី ៣: កំណត់រចនាសម្ព័ន្ធ Model និងចាប់ផ្តើមបង្រៀន (LoRA Training)

In [ ]:
import torch
import dataclasses
from typing import Any, Dict, List, Union
from datasets import load_dataset, Audio
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate

MODEL_ID = "openai/whisper-small"
OUTPUT_LORA_DIR = "./khmer_whisper_lora_output"

# 1. Load Preprocessor
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_ID)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_ID, language="Khmer", task="transcribe")
processor = WhisperProcessor.from_pretrained(MODEL_ID, language="Khmer", task="transcribe")

# 2. Load Base Whisper Model in FP16
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# 3. Configure LoRA Parameters
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
print("LoRA adapter initialized successfully!")

## ជំហានទី ៤: Merge LoRA Weights & Export to CTranslate2 (int8)
បម្លែង Model ឱ្យទៅជា **CTranslate2 int8** ដើម្បីយកទៅ Run លើ CPU Server របស់ KhmerSub ដោយល្បឿនលឿនបំផុត!

In [ ]:
!mkdir -p ./khmer_whisper_merged
!mkdir -p ./khmer_whisper_ct2_int8

# Convert directly using CTranslate2
# !ct2-transformers-converter --model ./khmer_whisper_merged --output_dir ./khmer_whisper_ct2_int8 --quantization int8
print("Export commands ready!")